# Step3B Rerank

In [ ]:
%pip install -q "datasets==3.6.0" "transformers>=4.46,<5" sentence-transformers rank_bm25 accelerate sentencepiece protobuf scikit-learn pandas pyarrow

In [ ]:
import os, re, json, joblib, collections, numpy as np, pandas as pd, torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/veritas"
except ImportError:
    DRIVE = "veritas_local"
OUT = f"{DRIVE}/retrieval"; os.makedirs(OUT, exist_ok=True)
MODEL_DIR = f"{DRIVE}/verdict_model_v2"
EMB_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
TOPK = 5
NCAND = 30
RERANK_MODEL = "BAAI/bge-reranker-base"
KS = (1, 5, 10)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["SUPPORTS", "REFUTES", "NEI"]
print("device:", DEV, "| model dir exists:", os.path.isdir(MODEL_DIR))

In [ ]:
sf_corpus = load_dataset("allenai/scifact", "corpus", trust_remote_code=True)["train"]
hv_all = load_dataset("dwadden/healthver_entailment", trust_remote_code=True)

docs = {}
for d in sf_corpus:
    docs[f"sf_{d['doc_id']}"] = dict(source="scifact", title=d["title"] or "", text=" ".join(d["abstract"]).strip())
for sp in hv_all:
    for ex in hv_all[sp]:
        docs.setdefault(f"hv_{ex['abstract_id']}", dict(source="healthver", title=ex["title"] or "",
                                                         text=" ".join(ex["abstract"]).strip()))
corpus = pd.DataFrame([dict(uid=u, **v) for u, v in docs.items()])
corpus["search_text"] = corpus["title"] + ". " + corpus["text"]
print(corpus.source.value_counts().to_dict(), "| total:", len(corpus))
print("median abstract words:", int(corpus.text.str.split().str.len().median()))

In [ ]:
def build_claims(rows):
    by = collections.OrderedDict()
    for cid, claim, uid, lab in rows:
        e = by.setdefault(cid, dict(claim=claim.strip(), gold=set(), labs=[]))
        if uid is not None and lab in ("SUPPORTS", "REFUTES"):
            e["gold"].add(uid); e["labs"].append(lab)
    out = []
    for e in by.values():
        lab = collections.Counter(e["labs"]).most_common(1)[0][0] if e["labs"] else "NEI"
        out.append(dict(claim=e["claim"], gold=e["gold"], label=LABELS.index(lab)))
    return out

sf = load_dataset("allenai/scifact", "claims", trust_remote_code=True)["validation"]
m = {"SUPPORT": "SUPPORTS", "CONTRADICT": "REFUTES"}
sf_rows = [(ex["id"], ex["claim"], f"sf_{ex['evidence_doc_id']}" if ex["evidence_label"] else None, m.get(ex["evidence_label"])) for ex in sf]

hm = {"SUPPORT": "SUPPORTS", "CONTRADICT": "REFUTES", "NEI": "NEI"}
def hv_rows(split):
    return [(ex["claim_id"], ex["claim"], f"hv_{ex['abstract_id']}", hm.get(ex["verdict"])) for ex in hv_all[split]]

EVAL = {"SciFact val": build_claims(sf_rows),
        "HealthVer val": build_claims(hv_rows("validation")),
        "HealthVer test": build_claims(hv_rows("test"))}
for k, v in EVAL.items():
    n_gold = sum(bool(c["gold"]) for c in v)
    print(f"{k:15s} claims: {len(v):4d} | with gold evidence: {n_gold:4d} | NEI-only: {len(v) - n_gold:4d}")

In [ ]:
tok_re = re.compile(r"[a-z0-9]+")
STOP = set("a an and are as at be by for from has have in is it its of on or that the this to was were will with".split())
def toks(t): return [w for w in tok_re.findall(t.lower()) if w not in STOP]

bm25 = BM25Okapi([toks(t) for t in corpus.search_text])

st = SentenceTransformer(EMB_MODEL, device=DEV); st.max_seq_length = 256
emb_path = f"{OUT}/corpus_emb.npy"
if os.path.exists(emb_path) and len(np.load(emb_path, mmap_mode="r")) == len(corpus):
    D = np.load(emb_path)
else:
    D = st.encode(corpus.search_text.tolist(), batch_size=64, normalize_embeddings=True,
                  show_progress_bar=True, convert_to_numpy=True).astype("float32")
print("doc embeddings:", D.shape)

def rank_bm25(queries, n=100):
    return [np.argsort(-bm25.get_scores(toks(q)))[:n] for q in queries]

def rank_dense(queries, n=100):
    Q = st.encode([QUERY_PREFIX + q for q in queries], batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    S = Q @ D.T
    return [np.argsort(-s)[:n] for s in S]

def fuse(r1, r2, n=100, k=60):
    out = []
    for a, b in zip(r1, r2):
        sc = collections.defaultdict(float)
        for r in (a, b):
            for rank, i in enumerate(r): sc[i] += 1.0 / (k + rank + 1)
        out.append(np.array(sorted(sc, key=sc.get, reverse=True)[:n]))
    return out

uid_of = corpus.uid.values

In [ ]:
RANKS = {}
rows = []
for name, claims in EVAL.items():
    qs = [c["claim"] for c in claims]
    rb, rd = rank_bm25(qs), rank_dense(qs)
    RANKS[name] = {"BM25": rb, "Dense": rd, "Hybrid": fuse(rb, rd)}
    has_gold = [i for i, c in enumerate(claims) if c["gold"]]
    for meth, R in RANKS[name].items():
        rec = {k: np.mean([bool(set(uid_of[R[i][:k]]) & claims[i]["gold"]) for i in has_gold]) for k in KS}
        rows.append(dict(set=name, method=meth, **{f"recall@{k}": round(v, 3) for k, v in rec.items()}))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
from sentence_transformers import CrossEncoder
rr = CrossEncoder(RERANK_MODEL, max_length=384, device=DEV)
if DEV == "cuda": rr.model.half()

def rerank(claims, rankings, ncand=NCAND, bs=64):
    pairs = [(c["claim"], corpus.search_text.iat[di]) for c, r in zip(claims, rankings) for di in r[:ncand]]
    sc = np.asarray(rr.predict(pairs, batch_size=bs, show_progress_bar=False), dtype="float64").reshape(len(claims), -1)
    if sc.min() < 0 or sc.max() > 1: sc = 1 / (1 + np.exp(-sc))
    new_r, new_s = [], []
    for r, s in zip(rankings, sc):
        o = np.argsort(-s); new_r.append(r[:ncand][o]); new_s.append(s[o])
    return new_r, np.array(new_s)

RERANKED, RSCORE = {}, {}
for name, claims in EVAL.items():
    RERANKED[name], RSCORE[name] = rerank(claims, RANKS[name]["Hybrid"])

rows = []
for name, claims in EVAL.items():
    has_gold = [i for i, c in enumerate(claims) if c["gold"]]
    for meth, R, ks in [("Hybrid", RANKS[name]["Hybrid"], (1, 5, 10, NCAND)), ("Hybrid+rerank", RERANKED[name], (1, 5, 10))]:
        rec = {k: np.mean([bool(set(uid_of[R[i][:k]]) & claims[i]["gold"]) for i in has_gold]) for k in ks}
        rows.append(dict(set=name, method=meth, **{f"recall@{k}": round(v, 3) for k, v in rec.items()}))
print(pd.DataFrame(rows).astype(object).fillna("").to_string(index=False))
print("(recall@30 on the Hybrid row is the ceiling: the reranker can't find what search didn't fetch)")

In [ ]:
vtok = AutoTokenizer.from_pretrained(MODEL_DIR)
vmodel = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEV).eval()
if DEV == "cuda": vmodel = vmodel.half()

@torch.no_grad()
def verdict_probs(pairs, bs=32, max_len=384):
    out = []
    for i in range(0, len(pairs), bs):
        b = pairs[i:i + bs]
        enc = vtok([c for c, _ in b], [d for _, d in b], truncation="only_second", max_length=max_len,
                   padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(vmodel(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

def pair_probs(claims, rankings, topk=TOPK):
    pairs = [(c["claim"], corpus.text.iat[di]) for c, r in zip(claims, rankings) for di in r[:topk]]
    return verdict_probs(pairs).reshape(len(claims), topk, 3)

P_HYB = {n: pair_probs(c, RANKS[n]["Hybrid"]) for n, c in EVAL.items()}
P_RR  = {n: pair_probs(c, RERANKED[n]) for n, c in EVAL.items()}
Y = {n: np.array([c["label"] for c in cl]) for n, cl in EVAL.items()}
print({n: P_RR[n].shape for n in EVAL})

In [ ]:
def decide(s, r, tau):
    return np.where(np.maximum(s, r) < tau, 2, np.where(s >= r, 0, 1))
def max_rule(P): return P[:, :, 0].max(1), P[:, :, 1].max(1)
F1 = lambda y, p: f1_score(y, p, average="macro")

VAL = ["SciFact val", "HealthVer val"]; TEST = "HealthVer test"
GRID = np.round(np.arange(0.3, 0.951, 0.05), 2)
def pick_tau(Pd):
    sc = [np.mean([F1(Y[n], decide(*max_rule(Pd[n]), t)) for n in VAL]) for t in GRID]
    return GRID[int(np.argmax(sc))], max(sc)
tau_hyb, _ = pick_tau(P_HYB); tau_rr, _ = pick_tau(P_RR)
print(f"tau chosen on validation only -> hybrid: {tau_hyb}, reranked: {tau_rr}")

def make_feats(P, S):
    S = S[:, :P.shape[1]]
    w = S / (S.sum(1, keepdims=True) + 1e-9)
    wp = (w[:, :, None] * P).sum(1)
    return np.column_stack([P[:, :, 0].max(1), P[:, :, 1].max(1), P[:, :, 0].mean(1), P[:, :, 1].mean(1),
                            P[:, 0, 0], P[:, 0, 1], P[:, 0, 2], P[:, :, 2].mean(1),
                            wp, S[:, 0], S[:, 0] - S[:, 1], S.mean(1)])

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

X = {n: make_feats(P_RR[n], RSCORE[n]) for n in EVAL}
Xv = np.vstack([X[n] for n in VAL]); yv = np.concatenate([Y[n] for n in VAL])
strat = np.concatenate([Y[n] + 3 * i for i, n in enumerate(VAL)])
mk = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000))
cv_pred = cross_val_predict(mk(), Xv, yv, cv=StratifiedKFold(5, shuffle=True, random_state=0).split(Xv, strat))
agg = mk().fit(Xv, yv)
offs = np.cumsum([0] + [len(Y[n]) for n in VAL])

rows = []
def add(label, f):
    rows.append({"method": label, **{n: round(f(n), 3) for n in EVAL}})
add("A. step 3: hybrid top-5, max rule", lambda n: F1(Y[n], decide(*max_rule(P_HYB[n]), tau_hyb)))
add("B. rerank top-5, max rule",         lambda n: F1(Y[n], decide(*max_rule(P_RR[n]), tau_rr)))
add("C. rerank top-5, learned aggregator", lambda n: F1(Y[n], cv_pred[offs[VAL.index(n)]:offs[VAL.index(n)+1]]) if n in VAL
                                                      else F1(Y[n], agg.predict(X[n])))
add("(ceiling) gold abstract given, step 2", lambda n: np.nan)
res = pd.DataFrame(rows); res.loc[3, "HealthVer test"] = 0.705
print("End-to-end macro-F1. Val columns for C are 5-fold cross-validated; HealthVer test is scored once.")
print(res.to_string(index=False))

In [ ]:
pred = agg.predict(X[TEST]); y = Y[TEST]
print("HealthVer test, learned aggregator (rows = true, cols = pred):")
print(pd.crosstab(pd.Series([LABELS[i] for i in y], name="true"), pd.Series([LABELS[i] for i in pred], name="pred")))
pa = decide(*max_rule(P_HYB[TEST]), tau_hyb)
nei = y == 2
print(f"\ntrue-NEI claims called SUPPORTS: step 3 = {int((pa[nei]==0).sum())}/{int(nei.sum())}  ->  now = {int((pred[nei]==0).sum())}/{int(nei.sum())}")

proba = agg.predict_proba(X[TEST]); conf = proba.max(1)
print("\nSelective prediction on HealthVer test (answer only when confident):")
rows = [dict(min_confidence=t, answered=f"{(conf >= t).mean():.0%}", accuracy_on_answered=round((pred[conf >= t] == y[conf >= t]).mean(), 3))
        for t in (0.0, 0.5, 0.6, 0.7, 0.8)]
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
def check_claim(text, show=3):
    c = [dict(claim=text)]
    cand = fuse(rank_bm25([text]), rank_dense([text]))
    rk, S = rerank(c, cand)
    P = pair_probs(c, rk, TOPK)
    p = agg.predict_proba(make_feats(P, S))[0]
    order = list(agg.classes_)
    probs = {LABELS[k]: float(p[order.index(k)]) if k in order else 0.0 for k in range(3)}
    verdict = max(probs, key=probs.get)
    evid = [dict(uid=uid_of[di], title=corpus.title.iat[di], rerank=float(S[0, j]),
                 passage_verdict=LABELS[int(P[0, j].argmax())], passage_probs=P[0, j].round(3).tolist())
            for j, di in enumerate(rk[0][:show])]
    return dict(claim=text, verdict=verdict, confidence=round(probs[verdict], 3), probs=probs, evidence=evid)

for t in ["Vitamin D supplementation reduces the risk of respiratory infections.",
          "Masks do not reduce the spread of COVID-19.",
          "Drinking lemon water cures cancer."]:
    r = check_claim(t)
    print(f"\n{r['claim']}\n  -> {r['verdict']} ({r['confidence']})  {r['probs']}")
    for e in r["evidence"]: print(f"     [{e['passage_verdict']:8s}] {e['rerank']:.2f}  {e['title'][:90]}")

In [ ]:
corpus.to_parquet(f"{OUT}/corpus.parquet", index=False)
np.save(emb_path, D)
joblib.dump(agg, f"{OUT}/aggregator.joblib")
json.dump(dict(emb_model=EMB_MODEL, query_prefix=QUERY_PREFIX, topk=TOPK, ncand=NCAND, rerank_model=RERANK_MODEL,
               n_docs=len(corpus), tau_max_rule=float(tau_rr),
               feature_order="see make_feats in step3b notebook"),
          open(f"{OUT}/retrieval_config.json", "w"))
print(sorted(os.listdir(OUT)))